# Machine Learning Methylation Experiment
___
Here, we'll narrow down PPMI project 120's m-values to the 450k range of CpG's and train an XGBoost classifier. Afterwards, we'll test the model against GSE111629.

In [ ]:
import pandas as pd
src_data_path = "/root/data/ppmi/data_ppmi.parquet"
df_ppmi = pd.read_parquet(src_data_path)
df_ppmi = df_ppmi[df_ppmi["Sample_Group"].isin(["Control", "PD"])]

In [34]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from cuml.preprocessing import StandardScaler
import optuna
from sklearn.model_selection import StratifiedKFold, train_test_split
from cuml.linear_model import LogisticRegression
import xgboost as xgb
import gc
from sklearn.metrics import average_precision_score, roc_auc_score

def cv_pipeline(X_all, Y_all):
    outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    fold_aucs_roc = []
    fold_aucs_pr = []
    variances = np.var(X_all, axis=0)
    threshold = np.percentile(variances, 75) 
    high_var_mask = variances > threshold
    X_high_var = X_all[:, high_var_mask]
    print(f"Reduced features from {X_all.shape[1]} to {X_high_var.shape[1]} based on variance.")

    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X_high_var, Y_all)):
        print(f"\n=================== FOLD {fold + 1} ===================")
        
        X_train, Y_train = X_high_var[train_idx], Y_all[train_idx]
        X_test, Y_test = X_high_var[test_idx], Y_all[test_idx]
        
        print("Feature Selection...")
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled  = scaler.transform(X_test)

        selector = SelectKBest(score_func=f_classif, k=500)
        if hasattr(X_train_scaled, 'get'): 
            X_train_cpu = X_train_scaled.get()
            Y_train_cpu = Y_train.get()
            X_test_cpu = X_test_scaled.get()
        else:
            X_train_cpu = X_train_scaled
            Y_train_cpu = Y_train
            X_test_cpu = X_test_scaled

        selector.fit(X_train_cpu, Y_train_cpu)

        X_train_sel = selector.transform(X_train_cpu)
        X_test_sel  = selector.transform(X_test_cpu)
        
        model = xgb.XGBClassifier(
            tree_method="hist", 
            device="cuda",
            objective="binary:logistic",
            n_estimators=200,
            max_depth=2,
            learning_rate=0.05,
            colsample_bytree=0.3,
            subsample=0.8,
            random_state=42,
            scale_pos_weight=(Y_train_cpu == 0).sum() / (Y_train_cpu == 1).sum()
        )
    
        model.fit(X_train_sel, Y_train_cpu, verbose=False)

        preds = model.predict_proba(X_test_sel)[:, 1]
    
        auc_roc = roc_auc_score(Y_test, preds)
        auc_pr = average_precision_score(Y_test, preds)
        
        fold_aucs_roc.append(auc_roc)
        fold_aucs_pr.append(auc_pr)

        print(f"ROC-AUC: {auc_roc:.4f} | PR-AUC: {auc_pr:.4f}")

        del model, selector
        gc.collect()

    print("\n================ FINAL RESULTS ================")
    print(f"Rigorous Cross-Validated ROC-AUC: {np.mean(fold_aucs_roc):.4f} ± {np.std(fold_aucs_roc):.4f}")
    print(f"Rigorous Cross-Validated PR-AUC: {np.mean(fold_aucs_pr):.4f} ± {np.std(fold_aucs_pr):.4f}")

In [43]:
target_var = "Sample_Group"
meta_cols_ppmi = [col for col in df_ppmi.columns if not (col.startswith("cg") or col.startswith("ch"))]
Y_pd_ppmi = df_ppmi[target_var]
X_pd_ppmi = df_ppmi.drop(columns=meta_cols_ppmi)

Y_pd_ppmi = Y_pd_ppmi.map({"Control": 0, "PD": 1})
X_ppmi = X_pd_ppmi.to_numpy(dtype="float32")
Y_ppmi = Y_pd_ppmi.to_numpy(dtype="float32")

cv_pipeline(X_ppmi, Y_ppmi)

Reduced features from 776947 to 194237 based on variance.

=================== FOLD 1 ===================
Feature Selection...
ROC-AUC: 0.5782 | PR-AUC: 0.7861

=================== FOLD 2 ===================
Feature Selection...
ROC-AUC: 0.5795 | PR-AUC: 0.7628

=================== FOLD 3 ===================
Feature Selection...
ROC-AUC: 0.6398 | PR-AUC: 0.8139

=================== FOLD 4 ===================
Feature Selection...
ROC-AUC: 0.5672 | PR-AUC: 0.7574

=================== FOLD 5 ===================
Feature Selection...
ROC-AUC: 0.6183 | PR-AUC: 0.8243

=================== FOLD 6 ===================
Feature Selection...
ROC-AUC: 0.6022 | PR-AUC: 0.8525

=================== FOLD 7 ===================
Feature Selection...
ROC-AUC: 0.5215 | PR-AUC: 0.7736

=================== FOLD 8 ===================
Feature Selection...
ROC-AUC: 0.6102 | PR-AUC: 0.8304

=================== FOLD 9 ===================
Feature Selection...
ROC-AUC: 0.7204 | PR-AUC: 0.8808

=================== FOL

In [12]:
df_ppmi["Sample_Group"].value_counts()

Sample_Group
PD         309
Control    122
Name: count, dtype: int64